# Part 0: Data Preparation

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
df = pd.read_csv("spam.csv", encoding="cp1252")

# Check dataset structure
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (5572, 5)
Columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
# Keep required columns
df = df[["v1", "v2"]].copy()

# Rename columns
df = df.rename(columns={
    "v1": "label",
    "v2": "text"
})

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicates
print("\nDuplicate rows:", df.duplicated().sum())

# Check original label distribution
print("\nLabel distribution:")
print(df["label"].value_counts())

Missing values:
label    0
text     0
dtype: int64

Duplicate rows: 403

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
# Encode labels
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

# Verify encoding
print("Label distribution:")
print(df["label"].value_counts())

print("\nMissing labels:", df["label"].isnull().sum())

df.head()

Label distribution:
label
0    4825
1     747
Name: count, dtype: int64

Missing labels: 0


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Create stratified train/test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training set size:", len(train_df))
print("Testing set size:", len(test_df))

Training set size: 4457
Testing set size: 1115


In [7]:
# Calculate spam rates
train_spam_rate = train_df["label"].mean() * 100
test_spam_rate = test_df["label"].mean() * 100

# Create summary table
stats = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Total Messages": [len(train_df), len(test_df)],
    "Spam Messages": [
        train_df["label"].sum(),
        test_df["label"].sum()
    ],
    "Ham Messages": [
        (train_df["label"] == 0).sum(),
        (test_df["label"] == 0).sum()
    ],
    "Spam Rate (%)": [
        round(train_spam_rate, 2),
        round(test_spam_rate, 2)
    ]
})

stats

,Dataset,Total Messages,Spam Messages,Ham Messages,Spam Rate (%)
0,Training,4457,598,3859,13.42
1,Testing,1115,149,966,13.36


In [8]:
# Save shared datasets
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Training and testing datasets saved successfully!")

Training and testing datasets saved successfully!


# Notes

We loaded the spam dataset and kept only the message text and label columns. We checked for missing values and duplicates, finding no missing values and 403 duplicate messages. We decided to keep the duplicates since repeated messages can occur in real spam data. We then encoded ham as 0 and spam as 1 and created an 80/20 stratified train/test split using random_state=42. This maintained similar spam proportions across both sets and ensured all three models would be evaluated using the same data. However, retaining duplicates meant that 115 distinct messages appeared in both the training and testing sets, which could lead to overly optimistic model performance since the models may have already encountered identical messages during training.